# LIAR DATASET

In [1]:
import kagglehub
import os
import shutil
import pandas as pd
import numpy as np

os.environ['OPENAI_KEY_SAE'] = '...'

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

from hypothesaes.quickstart import train_sae, interpret_sae, generate_hypotheses, evaluate_hypotheses
from hypothesaes.embedding import get_openai_embeddings, get_local_embeddings

In [2]:
# source_path = kagglehub.dataset_download("doanquanvietnamca/liar-dataset")
# current_dir = os.getcwd()
# destination_path = os.path.join(current_dir, "liar_data")
# shutil.move(source_path, destination_path)

In [3]:
column_names = ['id', 'label', 'statement', 'subject', 'speaker', 'title', 'state', 'party', 'barely_true', 'false', 'half_true', 'mostly_true', 'pants_on_fire', 'context']
def clean_label(lbl):
    if lbl in ['false', 'pants-fire']:
        return 0
    elif lbl in ['true', 'mostly-true']:
        return 1

def preprocess_liar_data(path):
    df = pd.read_csv(path, sep="\t", names=column_names)
    df["binary_label"] = df['label'].apply(clean_label)
    cleaned_df = df[["id", "statement", "binary_label"]].dropna()
    cleaned_df['binary_label'] = cleaned_df['binary_label'].astype(int)
    return cleaned_df

In [4]:
train_df = preprocess_liar_data('liar_data/train.tsv')
val_df = preprocess_liar_data('liar_data/valid.tsv')
test_df = preprocess_liar_data('liar_data/test.tsv')

In [5]:
texts = train_df['statement'].tolist()
labels = train_df['binary_label'].values

val_texts = val_df['statement'].tolist()
val_labels = val_df['binary_label'].values

test_texts = test_df['statement'].tolist()
test_labels = test_df['binary_label'].values

In [6]:
EMBEDDER = "text-embedding-3-small" # OpenAI
# EMBEDDER = "nomic-ai/modernbert-embed-base" # HuggingFace
CACHE_NAME = f"liar_quickstart_{EMBEDDER}"

text2embedding = get_openai_embeddings(texts + val_texts + test_texts, model=EMBEDDER, cache_name=CACHE_NAME)
# text2embedding = get_local_embeddings(texts + val_texts, model=EMBEDDER, cache_name=CACHE_NAME)

train_embeddings = np.stack([text2embedding[text] for text in texts])
val_embeddings = np.stack([text2embedding[text] for text in val_texts])
test_embeddings = np.stack([text2embedding[text] for text in test_texts])

Loading embedding chunks:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded 10399 embeddings in 0.3s


In [7]:
train_embeddings

array([[ 0.0168424 ,  0.02819385,  0.02217494, ...,  0.02002344,
        -0.03920213,  0.04424429],
       [-0.00856844,  0.01108797,  0.04243419, ..., -0.02927551,
        -0.03043837, -0.01029233],
       [ 0.05474834,  0.02636961,  0.04949953, ...,  0.02930794,
        -0.02257741,  0.00123608],
       ...,
       [ 0.04425092, -0.01292477,  0.03207557, ...,  0.00799007,
        -0.01729452, -0.00762112],
       [ 0.01822323, -0.02242166,  0.0324844 , ...,  0.00754142,
        -0.01177362,  0.0108225 ],
       [ 0.02107765,  0.05016765,  0.04719532, ...,  0.01996626,
        -0.03822666,  0.01615393]], shape=(6472, 1536))

In [8]:
checkpoint_dir = os.path.join("checkpoints", CACHE_NAME)
sae = train_sae(embeddings=train_embeddings, val_embeddings=val_embeddings,
                M=256, K=8, matryoshka_prefix_lengths=[32, 256], 
                checkpoint_dir=checkpoint_dir)

Loaded model from checkpoints/liar_quickstart_text-embedding-3-small/SAE_matryoshka_M=256_K=8_prefixes=32-256.pt onto device cpu


In [9]:
model = LogisticRegression(random_state=23, C=0.3).fit(train_embeddings, labels)
train_results = model.predict(train_embeddings)
val_results = model.predict(val_embeddings)
test_results = model.predict(test_embeddings)

In [10]:
train_results

array([0, 1, 0, ..., 0, 0, 0], shape=(6472,))

In [11]:
def calculate_metrics(results, labels):
    tps = np.count_nonzero((results == labels) & (results == 1))
    fps = np.count_nonzero((results != labels) & (results == 1))
    fns = np.count_nonzero((results != labels) & (results == 0))
    accuracy = np.mean(results == labels) * 100
    precision = tps / (tps + fps)
    recall = tps / (tps + fns)
    f1 = 2 * precision*recall / (precision + recall)
    return "\n".join([format(accuracy, "Accuracy"), format(precision, "Precision"), format(recall, "Recall"), format(f1, "F1 Score")])

def format(num, label, rounding=2):
    return f"{label}: {round(num, rounding)}"

In [12]:
print("Train Results \n---------------")
print(calculate_metrics(train_results, labels))
print("\nValidation Results \n---------------")
print(calculate_metrics(val_results, val_labels))
print("\nTest Results \n---------------")
print(calculate_metrics(test_results, test_labels))

Train Results 
---------------
Accuracy: 70.89
Precision: 0.71
Recall: 0.81
F1 Score: 0.76

Validation Results 
---------------
Accuracy: 69.71
Precision: 0.68
Recall: 0.8
F1 Score: 0.73

Test Results 
---------------
Accuracy: 69.62
Precision: 0.7
Recall: 0.83
F1 Score: 0.76


In [13]:
mlp = MLPClassifier(solver='adam', alpha=1e-5, hidden_layer_sizes=(16,), early_stopping=True, random_state=23).fit(train_embeddings, labels)

In [14]:
train_results_mlp = mlp.predict(train_embeddings)
val_results_mlp = mlp.predict(val_embeddings)
test_results_mlp = mlp.predict(test_embeddings)

In [15]:
print("Train Results \n---------------")
print(calculate_metrics(train_results_mlp, labels))
print("\nValidation Results \n---------------")
print(calculate_metrics(val_results_mlp, val_labels))
print("\nTest Results \n---------------")
print(calculate_metrics(test_results_mlp, test_labels))

Train Results 
---------------
Accuracy: 69.95
Precision: 0.71
Recall: 0.8
F1 Score: 0.75

Validation Results 
---------------
Accuracy: 70.21
Precision: 0.69
Recall: 0.8
F1 Score: 0.74

Test Results 
---------------
Accuracy: 68.48
Precision: 0.69
Recall: 0.81
F1 Score: 0.75
